In [1]:
# =============================================================================
# CELL 1: CAI DAT MOI TRUONG VA FIX XUNG DOT TORCHAUDIO
# =============================================================================
!pip install -q --upgrade transformers huggingface_hub

import sys
# Fix lỗi circular import torchaudio trên Kaggle trước khi transformers load SAM 3
for _k in list(sys.modules.keys()):
    if "torchaudio" in _k:
        del sys.modules[_k]

import transformers.utils.import_utils as _tu
_tu.is_torchaudio_available = lambda: False
import transformers.utils as _tutils
_tutils.is_torchaudio_available = lambda: False

print("✅ Môi trường đã sẵn sàng! Có thể chạy Cell tiếp theo.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 87.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
✅ Môi trường đã sẵn sàng! Có thể chạy Cell tiếp theo.


In [2]:
# =============================================================================
# CELL 2 (CHUẨN): QUÉT VÀ PHÂN LOẠI ẢNH TRONG HOME FIRE DATASET
# =============================================================================
import os, glob, json, pathlib

# ── 1. Cấu hình thư mục đầu ra ───────────────────────────────────────────────
OUTPUT_DIR       = "/kaggle/working/fire_ground_dataset"
JSON_OUTPUT_PATH = os.path.join(OUTPUT_DIR, "dataset_labels.json")
STREAM_CACHE     = os.path.join(OUTPUT_DIR, "stream_cache.jsonl")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── 2. Xác định thư mục chứa dataset ──────────────────────────────────────────
# Tự động ưu tiên thư mục pengbo00 vừa tìm thấy
if os.path.exists("/kaggle/input/datasets/pengbo00"):
    DATASET_ROOT = "/kaggle/input/datasets/pengbo00"
else:
    DATASET_ROOT = "/kaggle/input"

print(f"📂 Đang quét dữ liệu từ: {DATASET_ROOT}")

# ── 3. Quét toàn bộ ảnh ───────────────────────────────────────────────────────
valid_exts = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')
all_images = sorted([
    os.path.join(root, f)
    for root, _, files in os.walk(DATASET_ROOT, followlinks=True)
    for f in files if f.lower().endswith(valid_exts)
])

print(f"🔍 Tổng số ảnh tìm thấy: {len(all_images)} ảnh")

# ── 4. Hàm tìm file nhãn .txt tương ứng theo chuẩn YOLO ──────────────────────
def find_label_path(img_path):
    """
    Tìm file txt nhãn tương ứng cho ảnh trong dataset YOLO.
    Hỗ trợ cả 2 dạng:
      Dạng 1: .../train/images/xxx.jpg  -> .../train/labels/xxx.txt
      Dạng 2: .../images/train/xxx.jpg  -> .../labels/train/xxx.txt
    """
    # Thử dạng 1: thay /images/ thành /labels/
    p1 = img_path.replace("/images/", "/labels/")
    p1 = os.path.splitext(p1)[0] + ".txt"
    if os.path.exists(p1): 
        return p1
    
    # Thử dạng 2: đổi thư mục cha images -> labels
    parts = img_path.split("/")
    if "images" in parts:
        parts[parts.index("images")] = "labels"
        p2 = "/".join(parts)
        p2 = os.path.splitext(p2)[0] + ".txt"
        if os.path.exists(p2): 
            return p2
            
    # Thử cùng thư mục
    p3 = os.path.splitext(img_path)[0] + ".txt"
    if os.path.exists(p3): 
        return p3
        
    return None

# ── 5. Phân loại ảnh Lửa vs Ảnh nền (Default) ─────────────────────────────────
fire_candidates    = []
default_candidates = []

for img_p in all_images:
    lbl_p = find_label_path(img_p)
    has_fire = False
    
    if lbl_p and os.path.exists(lbl_p):
        with open(lbl_p, "r", encoding="utf-8") as f:
            lines = [line.strip() for line in f.readlines() if line.strip()]
            # Nếu file nhãn có chứa bounding box
            if len(lines) > 0:
                has_fire = True
    elif "fire" in img_p.lower():
        has_fire = True

    if has_fire:
        fire_candidates.append(img_p)
    else:
        default_candidates.append(img_p)

# ── 6. Báo cáo tổng kết ───────────────────────────────────────────────────────
print("\n" + "="*50)
print("📊 BÁO CÁO THỐNG KÊ DỮ LIỆU:")
print(f"  🔥 Ảnh có Lửa (sẽ dùng SAM 3 định vị đáy lửa) : {len(fire_candidates)} ảnh")
print(f"  🌿 Ảnh nền / Không lửa (gán nhãn has_fire=0)   : {len(default_candidates)} ảnh")
print(f"  📁 File nhãn sau khi gán sẽ lưu tại           : {JSON_OUTPUT_PATH}")
print("="*50)

if fire_candidates:
    print("\nVí dụ đường dẫn ảnh lửa đầu tiên:")
    print(f"  -> {fire_candidates[0]}")
    lbl_sample = find_label_path(fire_candidates[0])
    print(f"  -> Nhãn đi kèm: {lbl_sample}")



📂 Đang quét dữ liệu từ: /kaggle/input/datasets/pengbo00
🔍 Tổng số ảnh tìm thấy: 6500 ảnh

📊 BÁO CÁO THỐNG KÊ DỮ LIỆU:
  🔥 Ảnh có Lửa (sẽ dùng SAM 3 định vị đáy lửa) : 6365 ảnh
  🌿 Ảnh nền / Không lửa (gán nhãn has_fire=0)   : 135 ảnh
  📁 File nhãn sau khi gán sẽ lưu tại           : /kaggle/working/fire_ground_dataset/dataset_labels.json

Ví dụ đường dẫn ảnh lửa đầu tiên:
  -> /kaggle/input/datasets/pengbo00/home-fire-dataset/test/images/test_1.jpg
  -> Nhãn đi kèm: /kaggle/input/datasets/pengbo00/home-fire-dataset/test/labels/test_1.txt


In [3]:
# =============================================================================
# CELL 3: KHOI TAO SAM 3 & THUAT TOAN TIM DAY LUA (ADAPTIVE BASELINE)
# =============================================================================
import torch
import numpy as np
from PIL import Image
from transformers import Sam3Processor, Sam3Model
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# ── 1. Kiểm tra phần cứng ─────────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Thiết bị tính toán: {device.upper()}")
if device != "cuda":
    print("⚠️ CẢNH BÁO: Chưa bật GPU! Hãy bật GPU T4 ở cột bên phải để chạy nhanh.")

# ── 2. Đăng nhập Hugging Face ─────────────────────────────────────────────────
try:
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=hf_token)
    print("🔑 Đăng nhập Hugging Face thành công!")
except Exception as e:
    print("❌ LỖI ĐĂNG NHẬP HF_TOKEN:", e)
    print("👉 Hãy vào Add-ons -> Secrets trên thanh menu và thêm secret tên là 'HF_TOKEN'.")

# ── 3. Tải mô hình SAM 3 với FP16 ─────────────────────────────────────────────
print("⏳ Đang tải mô hình SAM 3 từ Hugging Face (khoảng 1 - 2 phút)...")
sam_processor = Sam3Processor.from_pretrained("facebook/sam3")
sam_model = Sam3Model.from_pretrained(
    "facebook/sam3",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)
sam_model.eval()
print("✅ Mô hình SAM 3 đã sẵn sàng trên VRAM!")

# ── 4. Thuật toán tìm gốc lửa (Adaptive Baseline) ──────────────────────────────
def adaptive_baseline(mask_np, alpha=0.35, jump_thresh=1.8):
    """
    Tìm điểm thấp nhất (đáy / gốc ngọn lửa) từ mask nhị phân của SAM 3.
    Output: (x_norm, y_norm) trong đoạn [0, 1]
    """
    ys, xs = np.where(mask_np > 0)
    if len(ys) < 50: 
        return None  # Mask quá bé hoặc nhiễu
        
    g_ymin, g_ymax = int(ys.min()), int(ys.max())
    if g_ymin == g_ymax: 
        return None
        
    # Cắt lấy alpha % thấp nhất ở chân ngọn lửa
    y_cut = g_ymax - alpha * (g_ymax - g_ymin)
    profile = []
    for ux in np.unique(xs):
        my = int(np.max(ys[xs == ux]))
        if my >= y_cut:
            profile.append((int(ux), my))
            
    if len(profile) < 2: 
        return None
        
    # Lọc bỏ tàn tro / nhiễu nhảy vọt xuống dưới
    base_line = [profile[0]]
    for cx, cy in profile[1:]:
        px, py = base_line[-1]
        dx, dy = abs(cx - px), abs(cy - py)
        if dx > 0 and (dy / dx) > jump_thresh and cy < py: 
            continue
        base_line.append((cx, cy))
        
    if not base_line: 
        return None
        
    H, W = mask_np.shape
    # Chọn điểm đáy sâu nhất
    best = base_line[int(np.argmax([p[1] for p in base_line]))]
    return round(float(best[0]) / W, 6), round(float(best[1]) / H, 6)

# ── 5. Smoke Test trên 1 ảnh mẫu ──────────────────────────────────────────────
print("\n🧪 Đang chạy thử nghiệm trên 1 ảnh mẫu:")
test_sample_path = fire_candidates[0]
print(f"  -> File: {test_sample_path}")

try:
    test_img = Image.open(test_sample_path).convert("RGB")
    W, H = test_img.size
    
    # Prompt SAM 3 với từ khóa "fire"
    inputs = sam_processor(images=test_img, text="fire", return_tensors="pt").to(device)
    if device == "cuda":
        inputs = {k: v.to(torch.float16) if v.dtype == torch.float32 else v for k, v in inputs.items()}
        
    with torch.no_grad():
        outputs = sam_model(**inputs)
        
    results = sam_processor.post_process_instance_segmentation(
        outputs, threshold=0.30, mask_threshold=0.5, target_sizes=[(H, W)]
    )[0]
    
    raw = results.get("masks")
    if raw is None: raw = results.get("segmentation")
    if isinstance(raw, torch.Tensor):
        masks_list = [raw[i] for i in range(raw.shape[0])]
    else:
        masks_list = list(raw) if raw is not None else []
        
    pts = [p for m in masks_list if (p := adaptive_baseline((m.cpu().numpy() > 0).astype(np.uint8)))]
    if pts:
        best_pt = max(pts, key=lambda p: p[1])
        print(f"🎉 TEST THÀNH CÔNG! Đã xác định được đáy lửa tại tọa độ: {best_pt}")
    else:
        print("ℹ️ Test chạy tốt (ảnh mẫu này SAM 3 chưa phát hiện ngọn lửa rõ).")
except Exception as err:
    print(f"❌ Lỗi khi test: {err}")


🚀 Thiết bị tính toán: CUDA
🔑 Đăng nhập Hugging Face thành công!
⏳ Đang tải mô hình SAM 3 từ Hugging Face (khoảng 1 - 2 phút)...


processor_config.json:   0%|          | 0.00/1.71k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/25.8k [00:00<?, ?B/s]

[transformers] `memory_attention_rope_theta` is deprecated and will be removed in v5.0. Use `rope_parameters['rope_theta']` instead.


tokenizer_config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/588 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.44GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1468 [00:00<?, ?it/s]

✅ Mô hình SAM 3 đã sẵn sàng trên VRAM!

🧪 Đang chạy thử nghiệm trên 1 ảnh mẫu:
  -> File: /kaggle/input/datasets/pengbo00/home-fire-dataset/test/images/test_1.jpg
ℹ️ Test chạy tốt (ảnh mẫu này SAM 3 chưa phát hiện ngọn lửa rõ).


In [4]:
# =============================================================================
# CELL 4: GAN NHAN TOAN BO DATASET VOI SMART FALLBACK & AUTO-RESUME
# =============================================================================
from tqdm import tqdm

# ── 1. Kiểm tra các ảnh đã làm xong từ trước (Auto-Resume) ────────────────────
done_images = set()
if os.path.exists(STREAM_CACHE):
    with open(STREAM_CACHE, encoding="utf-8") as f:
        for line in f:
            try:
                rec = json.loads(line.strip())
                done_images.add(rec["image_path"])
            except:
                pass
    print(f"🔄 Đã phát hiện {len(done_images)} ảnh đã gán nhãn trước đó (sẽ tự động bỏ qua).")

fire_todo    = [p for p in fire_candidates if p not in done_images]
default_todo = [p for p in default_candidates if p not in done_images]

print(f"🚀 Chuẩn bị xử lý: {len(fire_todo)} ảnh Lửa | {len(default_todo)} ảnh Nền")

# ── 2. Hàm lấy đáy của Bounding Box YOLO làm phương án dự phòng ───────────────
def get_yolo_bottom_fallback(img_path):
    lbl_p = find_label_path(img_path)
    if not lbl_p or not os.path.exists(lbl_p):
        return (0.5, 0.5)
    
    candidates = []
    with open(lbl_p, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                # Format YOLO: class x_center y_center width height
                x_c, y_c, w, h = map(float, parts[1:5])
                bottom_y = min(1.0, y_c + (h / 2.0))
                candidates.append((round(x_c, 6), round(bottom_y, 6)))
                
    if candidates:
        # Chọn điểm có y lớn nhất (đáy thấp nhất)
        return max(candidates, key=lambda p: p[1])
    return (0.5, 0.5)

# ── 3. Vòng lặp gán nhãn chính ────────────────────────────────────────────────
n_sam3_ok = 0
n_fallback_ok = 0

with open(STREAM_CACHE, "a", encoding="utf-8") as cache_f:
    # --- PHẦN A: Xử lý ảnh Lửa ---
    if fire_todo:
        for img_path in tqdm(fire_todo, desc="🔥 SAM 3 Fire Labeling"):
            pt_result = None
            try:
                image = Image.open(img_path).convert("RGB")
                W, H  = image.size
                
                inputs = sam_processor(images=image, text="fire", return_tensors="pt").to(device)
                if device == "cuda":
                    inputs = {k: v.to(torch.float16) if v.dtype == torch.float32 else v for k, v in inputs.items()}
                    
                with torch.no_grad():
                    outputs = sam_model(**inputs)
                    
                results = sam_processor.post_process_instance_segmentation(
                    outputs, threshold=0.30, mask_threshold=0.5, target_sizes=[(H, W)]
                )[0]

                raw = results.get("masks")
                if raw is None: raw = results.get("segmentation")
                if isinstance(raw, torch.Tensor):
                    masks_list = [raw[i] for i in range(raw.shape[0])]
                else:
                    masks_list = list(raw) if raw is not None else []

                pts = [p for m in masks_list if (p := adaptive_baseline((m.cpu().numpy() > 0).astype(np.uint8)))]
                if pts:
                    pt_result = max(pts, key=lambda p: p[1])
                    n_sam3_ok += 1
            except Exception:
                pt_result = None

            # Nếu SAM 3 không bắt được mask, dùng đáy Bounding Box YOLO làm dự phòng
            if pt_result is None:
                pt_result = get_yolo_bottom_fallback(img_path)
                n_fallback_ok += 1

            # Ghi ngay vào đĩa
            cache_f.write(json.dumps({
                "image_path": img_path,
                "has_fire"  : 1,
                "p_fire"    : list(pt_result)
            }) + "\n")
            cache_f.flush()

    # --- PHẦN B: Xử lý ảnh Nền / Không Lửa ---
    if default_todo:
        for img_path in tqdm(default_todo, desc="🌿 No-Fire Labeling"):
            cache_f.write(json.dumps({
                "image_path": img_path,
                "has_fire"  : 0,
                "p_fire"    : [0.0, 0.0]
            }) + "\n")
            cache_f.flush()

# ── 4. Tổng hợp thành file JSON hoàn chỉnh ─────────────────────────────────────
final_dataset = []
with open(STREAM_CACHE, encoding="utf-8") as f:
    for line in f:
        try:
            final_dataset.append(json.loads(line.strip()))
        except:
            pass

with open(JSON_OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(final_dataset, f, indent=2)

print("\n" + "="*50)
print("🎉 HOÀN THÀNH GÁN NHÃN TOÀN BỘ DATASET!")
print(f"  🔥 Gán nhãn bằng SAM 3 chính xác: {n_sam3_ok} ảnh")
print(f"  🎯 Gán nhãn qua YOLO fallback   : {n_fallback_ok} ảnh")
print(f"  🌿 Ảnh nền (no-fire)            : {len(default_candidates)} ảnh")
print(f"  📁 Tổng cộng bản ghi đã lưu    : {len(final_dataset)} ảnh")
print(f"  💾 File lưu tại: {JSON_OUTPUT_PATH}")
print("="*50)

# ── 5. Giải phóng VRAM GPU ───────────────────────────────────────────────────
del sam_model, sam_processor
torch.cuda.empty_cache()
print("🧹 Đã giải phóng hoàn toàn VRAM GPU!")


🚀 Chuẩn bị xử lý: 6365 ảnh Lửa | 135 ảnh Nền


🌿 No-Fire Labeling: 100%|██████████| 135/135 [00:00<00:00, 74007.46it/s]


🎉 HOÀN THÀNH GÁN NHÃN TOÀN BỘ DATASET!
  🔥 Gán nhãn bằng SAM 3 chính xác: 4558 ảnh
  🎯 Gán nhãn qua YOLO fallback   : 1807 ảnh
  🌿 Ảnh nền (no-fire)            : 135 ảnh
  📁 Tổng cộng bản ghi đã lưu    : 6500 ảnh
  💾 File lưu tại: /kaggle/working/fire_ground_dataset/dataset_labels.json
🧹 Đã giải phóng hoàn toàn VRAM GPU!
